# AutoSchemaKG — Concept Retrieval Research

Branch: `codex/research-concept-retrieval` (must be pushed before GitHub clone works).

Reuses a construction ZIP; does not rebuild the graph. `diagnostic` needs no GPU or Qwen server. `qa` needs GPU/Qwen. Default is **plan only**. Results persist on Drive. Three questions are a smoke test, not evidence of improvement.

See `RESEARCH.md` for hypotheses, controls, split policy and interpretation.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
import requests
from google.colab import drive, files
drive.mount('/content/drive')

RUN_ROOT = Path('/content/drive/MyDrive/AutoSchemaKG/research_en_001')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE = RUN_ROOT / 'construction.zip'  # Or a prepared VN input directory for bm25/dense.
STAGE = 'diagnostic'  # 'qa' enables answer generation and LLM edge filtering.
SPLIT = 'smoke'       # 'dev' for tuning; 'test' only after freezing choices.
EXECUTE = False      # First inspect the plan. Set True to run selected arms.
ALLOW_TEST = False
ARMS = ['bm25', 'dense', 'entity', 'entity_event', 'full', 'quota_20_10', 'factual_seeds']
DEV_SIZE = 100
SEED = 42
MODEL_ID = 'Qwen/Qwen3.5-2B'
LANGUAGE = 'en'  # 'vi' selects multilingual embeddings; runner verifies source metadata.
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small' if LANGUAGE == 'vi' else 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
PORT = 8000
CONTEXT_LENGTH = 4096
OUTPUT_DIR = RUN_ROOT / 'experiments'
BRANCH = 'codex/research-concept-retrieval'
assert STAGE in {'diagnostic', 'qa'}
if SPLIT == 'test' and EXECUTE and not ALLOW_TEST:
    raise ValueError('Freeze dev choices, then explicitly set ALLOW_TEST=True')
print('Persistent directory:', RUN_ROOT)


## Input

Upload the **construction/evaluated graph ZIP**, not `content.zip`. Existing input is never overwritten. For VN baselines before graph construction, point SOURCE to the prepared directory and select only `bm25`, `dense`.

In [ ]:
if not SOURCE.exists():
    uploaded = files.upload()
    candidates = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Upload exactly one construction ZIP')
    shutil.copy2(candidates[0], SOURCE)
print('Source:', SOURCE)


## Pin code and model revisions

This branch must already exist on your GitHub. This notebook never pushes code. On resume it restores the saved commit, not a newer branch head. Use a new RUN_ROOT when changing code/models/settings.

In [ ]:
REPO_DIR = Path('/content/SmallScaledAutoSchemaKG_research')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', str(REPO_DIR)], check=True)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
if status:
    raise RuntimeError('Clone contains edits. Preserve them before switching revisions.')
revision_file = RUN_ROOT / 'revisions.json'
if revision_file.exists():
    revisions = json.loads(revision_file.read_text())
    if revisions['model'] != MODEL_ID or revisions['embedding'] != EMBEDDING_MODEL:
        raise ValueError('Model changed: use a NEW RUN_ROOT')
    subprocess.run(['git', 'checkout', '--detach', revisions['git_commit']], cwd=REPO_DIR, check=True)
else:
    def model_revision(model):
        response = requests.get(f'https://huggingface.co/api/models/{model}', timeout=30)
        response.raise_for_status()
        return response.json()['sha']
    if not (REPO_DIR / 'scripts/run_research_experiments.py').exists():
        raise RuntimeError('Research files have not been pushed to this branch')
    revisions = {'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
                 'model': MODEL_ID, 'model_revision': model_revision(MODEL_ID),
                 'embedding': EMBEDDING_MODEL, 'embedding_revision': model_revision(EMBEDDING_MODEL)}
    revision_file.write_text(json.dumps(revisions, indent=2))
os.chdir(REPO_DIR)
print(json.dumps(revisions, indent=2))


## Isolated CPU client environment

Uses the existing v2 requirements. Installs dependencies but does not load Qwen weights. Keep dependency locks on Drive. Do not install vLLM into this environment.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
QA_ENV = '/content/autoschema_research_qa'
QA_PY = QA_ENV + '/bin/python'
if not Path(QA_PY).exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', QA_ENV], check=True)
qa_lock = RUN_ROOT / 'qa_requirements.lock.txt'
subprocess.run(['uv', 'pip', 'install', '--python', QA_PY, '--torch-backend=cpu', '-r',
                str(qa_lock) if qa_lock.exists() else 'requirements-hotpotqa-v2.txt'], check=True)
if not qa_lock.exists():
    qa_lock.write_text(subprocess.check_output(['uv', 'pip', 'freeze', '--python', QA_PY], text=True))


## Save and inspect the experiment plan

All diagnostic graph arms disable the LLM filter. QA arms enable it unless configured otherwise. Plan creation contacts no LLM. The suite, question splits, source fingerprint, package versions and code hashes are immutable within OUTPUT_DIR.

In [ ]:
command = [QA_PY, '-u', 'scripts/run_research_experiments.py', str(SOURCE),
    '--output-dir', str(OUTPUT_DIR), '--stage', STAGE, '--split', SPLIT,
    '--dev-size', str(DEV_SIZE), '--seed', str(SEED),
    '--model', MODEL_ID, '--model-revision', revisions['model_revision'],
    '--base-url', f'http://127.0.0.1:{PORT}/v1',
    '--embedding-model', EMBEDDING_MODEL, '--embedding-revision', revisions['embedding_revision'],
    '--embedding-device', 'cpu', '--context-length', str(CONTEXT_LENGTH)]
if ARMS:
    command += ['--arms'] + ARMS
if ALLOW_TEST:
    command += ['--allow-test']
subprocess.run(command, check=True)


## Qwen server — QA execution only

Skipped for CPU diagnostics and plan-only runs. If switching runtime to GPU resets /content, rerun setup from the top with the same Drive folder. Start-up logs and installed dependency versions are preserved. This uses the existing v2 server configuration; GPU/CUDA compatibility still needs validation on your runtime.

In [ ]:
if STAGE == 'qa' and EXECUTE:
    gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(gpu.stdout)
    if gpu.returncode:
        raise RuntimeError('Select a GPU runtime, then rerun setup')
    VLLM_ENV = '/content/autoschema_research_vllm'
    VLLM_PY = VLLM_ENV + '/bin/python'
    if not Path(VLLM_PY).exists():
        subprocess.run(['uv', 'venv', '--python', '3.12', VLLM_ENV], check=True)
    vllm_lock = RUN_ROOT / 'vllm_requirements.lock.txt'
    packages = ['-r', str(vllm_lock)] if vllm_lock.exists() else ['--pre', 'vllm']
    subprocess.run(['uv', 'pip', 'install', '--python', VLLM_PY, '--torch-backend=auto'] + packages, check=True)
    if not vllm_lock.exists():
        vllm_lock.write_text(subprocess.check_output(['uv', 'pip', 'freeze', '--python', VLLM_PY], text=True))
    LOG_PATH = RUN_ROOT / 'qwen_vllm.log'
    def ready():
        try:
            response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
            if response.ok:
                if MODEL_ID not in [m['id'] for m in response.json()['data']]:
                    raise RuntimeError('Port belongs to another model')
                return True
        except requests.RequestException:
            return False
        return False
    if not ready():
        server_log = open(LOG_PATH, 'a', encoding='utf-8')
        server = subprocess.Popen([VLLM_ENV + '/bin/vllm', 'serve', MODEL_ID,
            '--revision', revisions['model_revision'], '--host', '127.0.0.1', '--port', str(PORT),
            '--dtype', 'half', '--max-model-len', str(CONTEXT_LENGTH), '--max-num-seqs', '1',
            '--gpu-memory-utilization', '0.80', '--language-model-only'],
            stdout=server_log, stderr=subprocess.STDOUT)
        started = time.monotonic()
        while time.monotonic() - started < 1200:
            if server.poll() is not None:
                server_log.flush()
                print(LOG_PATH.read_text(errors='replace')[-12000:])
                raise RuntimeError('vLLM stopped; inspect the log above')
            if ready():
                break
            print(f'Waiting for Qwen: {time.monotonic() - started:.0f}s', flush=True)
            time.sleep(10)
        else:
            raise TimeoutError('Server not ready; inspect log, do not launch duplicate servers')
    print('Qwen ready. If reusing a server, ensure its weights match revisions.json.')
else:
    print('Qwen setup skipped')


## Execute / resume

Check ARMS and SPLIT before setting EXECUTE=True. Identical reruns skip completed questions. ETA is per arm and excludes setup/future arms. Never run two notebooks against the same OUTPUT_DIR.

In [ ]:
if EXECUTE:
    subprocess.run(command + ['--execute'], check=True)
else:
    print('Plan only. Set EXECUTE=True in configuration and rerun the plan/server/execution cells.')


## Report and archive

Complete runs only; oracle/no-context are kept separate. Markdown has the overview; JSON includes paired bootstrap, question-type breakdown, token usage and timings. Shared-cache setup timings are not cold-build comparisons. The ZIP below omits embedding caches (recomputable) but includes input, settings and results.

In [ ]:
if EXECUTE:
    subprocess.run([QA_PY, 'scripts/report_research_experiments.py', str(OUTPUT_DIR),
                    '--stage', STAGE, '--split', SPLIT], check=True)
    report_path = OUTPUT_DIR / f'report_{STAGE}_{SPLIT}.md'
    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text()))


In [ ]:
SAVE_ZIP = False  # Set True when ready to download; Drive remains the primary checkpoint store.
if SAVE_ZIP:
    from zipfile import ZipFile, ZIP_DEFLATED
    archive = Path('/content/autoschemakg_research.zip')
    with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as z:
        for path in RUN_ROOT.rglob('*'):
            if path.is_file() and 'embedding_cache' not in path.parts and path.name != '.research.lock':
                z.write(path, path.relative_to(RUN_ROOT))
    files.download(str(archive))
print('Persistent outputs:', RUN_ROOT)
